In [54]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

In [55]:
import matplotlib.pyplot as plt
import numpy as np
from monai.transforms import LoadImage, Resize, EnsureChannelFirst, Orientation, Compose, SaveImage, Transform
from monai.bundle import ConfigParser, download
from monai.data.meta_tensor import MetaTensor
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import SGD
import seaborn as sns
from ipywidgets import interact
import nibabel as nib
from nibabel import Nifti1Image
from totalsegmentator.python_api import totalsegmentator
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, random_split
import torch.optim as optim



In [56]:
%matplotlib inline

def show_scrollable_image(image, cmap="nipy_spectral"):
    """
    Displays a 3D array as a scrollable series of slices.
    
    Parameters:
    - image (numpy.ndarray): A 3D numpy array (e.g., (H, W, D)) representing the image.
    """
    
    # Define a function to display each slice
    def display_slice(slice_index):
        plt.figure(figsize=(8, 8))
        plt.imshow(image[:, :, slice_index], cmap=cmap)
        plt.colorbar(label='HU')
        plt.axis("off")
        plt.title(f"Slice {slice_index + 1} / {image.shape[2]}")
        plt.show()
    
    # Create an interactive slider to scroll through slices
    interact(display_slice, slice_index=(0, image.shape[2] - 1))

In [57]:
def nifti1image_to_metatensor(nifti_img: Nifti1Image) -> MetaTensor:
    """
    Converts a Nifti1Image object to a MetaTensor with metadata.
    Args:
        nifti_img (Nifti1Image): A loaded NIfTI image.
    Returns:
        MetaTensor: A MONAI MetaTensor containing the image data and metadata.
    """
    # Convert voxel data to PyTorch tensor
    image_tensor = torch.tensor(nifti_img.get_fdata(), dtype=torch.float32)

    # Extract metadata
    metadata = {
        "affine": torch.tensor(nifti_img.affine, dtype=torch.float32),  # Affine transformation matrix
        "spacing": nifti_img.header.get_zooms(),  # Voxel spacing (x, y, z)
        "original_shape": image_tensor.shape  # Store original shape for reference
    }

    # Create a MetaTensor with metadata
    meta_tensor = MetaTensor(image_tensor, meta=metadata)

    return meta_tensor

In [58]:
def resize_to_target_slices(image: MetaTensor, heart_mask: MetaTensor, target_width=512, target_height=512, target_slices=128):
    """
    Extracts the heart region from the input scan and resizes it to a target number of slices, 
    preserving metadata while correctly computing voxel spacing.

    Args:
        image (MetaTensor): The input 3D CT/MRI scan.
        heart_mask (MetaTensor): Binary mask indicating heart region.
        target_slices (int, optional): Number of slices to resize to. Default is 128.

    Returns:
        MetaTensor: Resized image with updated metadata.
    """
    if isinstance(heart_mask, MetaTensor):
        heart_mask = heart_mask.as_tensor().cpu().numpy()
    
    z_indices = np.any(heart_mask > 0, axis=(0, 1))
    heart_slices = np.where(z_indices)[0]
    # print(heart_slices)
    if len(heart_slices) == 0:
        raise ValueError("No heart region found in the mask.")
    
    start_slice = heart_slices[0]
    end_slice = heart_slices[-1] + 1
    
    sliced_image = image[:, :, start_slice:end_slice]
    if not isinstance(sliced_image, MetaTensor):
        sliced_image = MetaTensor(sliced_image, meta=image.meta)
    
    height, width, original_depth = image.shape
    # sliced_depth = sliced_image.shape[-1]

    sliced_image = sliced_image.unsqueeze(0)  # Shape becomes (1, H, W, Z)
    
    resizer = Resize(spatial_size=(target_width,target_height,target_slices), mode="trilinear", align_corners=True)
    resized_image = resizer(sliced_image)
    
    resized_image = resized_image.squeeze(0)

    # Update metadata
    new_meta = image.meta.copy()  
    
    original_spacing = image.meta.get("spacing", (1.0, 1.0, 1.0))
    new_spacing = (
        original_spacing[0],
        original_spacing[1],
        original_spacing[2] * (original_depth / target_slices)
    )
    new_meta["spacing"] = new_spacing

    # Keep affine transformation matrix but adjust slice resolution
    if "affine" in new_meta:
        new_meta["affine"][-1, -1] *= (original_depth / target_slices)  # Adjust Z scaling

    # Return resized MetaTensor with updated metadata
    return MetaTensor(resized_image, meta=new_meta)

In [59]:
def align_segmentation_to_image(segmentation):
    """
    Aligns the TotalSegmentator output (segmentation) with the original image
    by flipping the z-axis of the segmentation mask.
    
    Parameters:
        image (torch.Tensor or np.ndarray): Original image (H x W x D).
        segmentation (torch.Tensor or np.ndarray): TotalSegmentator output (H x W x D).
        
    Returns:
        torch.Tensor: Segmentation aligned with the image.
    """
   
    # Flip along the z-axis (last dimension)
    aligned_segmentation = torch.flip(segmentation, dims=[-1])
    
    return aligned_segmentation

# Example usage
# aligned_mask = align_segmentation_to_image(full_image)



In [60]:
def custom_name_formatter(metadict: dict, saver: Transform) -> dict:
    """Returns a kwargs dict for :py:meth:`FolderLayout.filename`,
    according to the input metadata and SaveImage transform."""
    subject = "heart_resized"
    patch_index = None
    return {"subject": f"{subject}", "idx": patch_index}

In [ ]:
def preprocess_full_heart(input_file_path: str):
    # input_folder = f"../data/Inputs/{input_file_path}"
    dicom_file_path = os.path.join(input_file_path, "DICOM")
    subfolders = sorted([f for f in os.listdir(dicom_file_path) if os.path.isdir(os.path.join(dicom_file_path, f))])
    
    if not subfolders:
        raise FileNotFoundError(f"No subfolders found in {dicom_file_path}")
    
    first_folder = subfolders[0]
    dicom_file_path = os.path.join(dicom_file_path, first_folder)    

    output_folder = input_file_path.replace("Inputs", "Outputs")
    
    if os.path.exists(f"{output_folder}/heart_resized.nii.gz"):
        image_loader = LoadImage(image_only=True, ensure_channel_first=True)
        resized_image = image_loader(f"{output_folder}/heart_resized.nii.gz")
        resized_image = resized_image.squeeze(dim=-1)
        # resized_image = resize_to_target_slices(output_image, output_image)
        # resized_image = nib.load(f"{output_folder}/heart_resized.nii.gz")
    else:
        # output_image = totalsegmentator(dicom_file_path, output_folder, license_number="aca_BWYHC6UQQFDU8A", roi_subset=["heart"], device="mps")
        output_image = totalsegmentator(dicom_file_path, output_folder, roi_subset=["heart"], device="mps")
        output_image = nifti1image_to_metatensor(output_image)
        resized_image = resize_to_target_slices(output_image, output_image,target_height=256,target_width=256,target_slices=64)
        # nib.save(resized_image, f"{output_folder}/heart_resized.nii.gz")
        print("saving")
        image_saver = SaveImage(output_dir=f"{output_folder}", separate_folder=False, output_postfix="", output_name_formatter=custom_name_formatter)
        image_saver(resized_image)
        print("saved")
        print(resized_image.shape)
    
    return resized_image
    


# preprocess_full_heart("takotsubo_cases/testing_subset/LET 04456356")


In [62]:
x_data = []
y_labels = []

In [63]:
takotsubo_data_path = '../data/Inputs/takotsubo_cases/'

# takotsubo_preprocessed_data = []

for folder_name in sorted(os.listdir(takotsubo_data_path)):
    folder_path = os.path.join(takotsubo_data_path, folder_name)
    if os.path.isdir(folder_path):
        print(folder_path)
        preprocessed_heart = preprocess_full_heart(folder_path)
        x_data.append(preprocessed_heart)
        y_labels.append(1)


../data/Inputs/takotsubo_cases/AD 12191953
torch.Size([256, 256, 64])
../data/Inputs/takotsubo_cases/AG 11370442
torch.Size([256, 256, 64])
../data/Inputs/takotsubo_cases/AMH 36922300
torch.Size([256, 256, 64])
../data/Inputs/takotsubo_cases/AMS 22722383
torch.Size([256, 256, 64])
../data/Inputs/takotsubo_cases/APH 03516218
torch.Size([256, 256, 64])
../data/Inputs/takotsubo_cases/BB 08903262
torch.Size([256, 256, 64])
../data/Inputs/takotsubo_cases/BEH 04037842 with thrombus
torch.Size([256, 256, 64])
../data/Inputs/takotsubo_cases/BFM 10624765
torch.Size([256, 256, 64])
../data/Inputs/takotsubo_cases/BLT 11022191
torch.Size([256, 256, 64])
../data/Inputs/takotsubo_cases/CAMcD 10747764 with thrombus
torch.Size([256, 256, 64])
../data/Inputs/takotsubo_cases/CDA 38093134
torch.Size([256, 256, 64])
../data/Inputs/takotsubo_cases/CLB 50002879
torch.Size([256, 256, 64])
../data/Inputs/takotsubo_cases/CML 11113354
torch.Size([256, 256, 64])
../data/Inputs/takotsubo_cases/CMS 04664231
torch.

In [64]:
normal_data_path = '../data/Inputs/normal_cases/'

# normal_preprocessed_data = []
for folder_name in sorted(os.listdir(normal_data_path)):
    folder_path = os.path.join(normal_data_path, folder_name)
    if os.path.isdir(folder_path):
        print(folder_path)
        preprocessed_heart = preprocess_full_heart(folder_path)
        x_data.append(preprocessed_heart)
        y_labels.append(0)

../data/Inputs/normal_cases/AAP 50415783_61F
torch.Size([256, 256, 64])
../data/Inputs/normal_cases/AB 10470318_90F
torch.Size([256, 256, 64])
../data/Inputs/normal_cases/AG 519880_37F
torch.Size([256, 256, 64])
../data/Inputs/normal_cases/AIB 36444628_88F
torch.Size([256, 256, 64])
../data/Inputs/normal_cases/AMB 08058471_74M
torch.Size([256, 256, 64])
../data/Inputs/normal_cases/ASJ 05633540_68F
torch.Size([256, 256, 64])
../data/Inputs/normal_cases/ASSA 39309711_56F
torch.Size([256, 256, 64])
../data/Inputs/normal_cases/BA 37586690_62M
torch.Size([256, 256, 64])
../data/Inputs/normal_cases/BJB 42898_78F
torch.Size([256, 256, 64])
../data/Inputs/normal_cases/BMB 26209098_65M
torch.Size([256, 256, 64])
../data/Inputs/normal_cases/BWB 08372559_71M
torch.Size([256, 256, 64])
../data/Inputs/normal_cases/CAB 30473847_51F
torch.Size([256, 256, 64])
../data/Inputs/normal_cases/CAD 35811017_49F
torch.Size([256, 256, 64])
../data/Inputs/normal_cases/CAM 29734621_78F
torch.Size([256, 256, 64])

In [65]:
class CTScanDataset(Dataset):
    def __init__(self, images, labels):
        self.images = images.float().unsqueeze(1)  # (N, 1, H, W, D)
        self.labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)  # (N, 1)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

# Convert MetaTensors to regular tensors
x_data = torch.stack([x.as_tensor() for x in x_data])  # (N, 512, 512, 128)
y_labels = torch.tensor(y_labels, dtype=torch.float32)

# Create dataset
dataset = CTScanDataset(x_data, y_labels)

# Split dataset (80% train, 20% test)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])


train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)


/var/folders/jm/k37qm_7n7dj5fp2rnwhd32y80000gn/T/ipykernel_82320/1091880197.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)  # (N, 1)


In [66]:
class CNN3D(nn.Module):
    def __init__(self):
        super(CNN3D, self).__init__()
        self.conv1 = nn.Conv3d(1, 16, kernel_size=3, stride=1, padding=1)  # Reduce to 16 filters
        self.pool = nn.MaxPool3d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv3d(16, 32, kernel_size=3, stride=1, padding=1)  # Reduce to 32 filters
        self.conv3 = nn.Conv3d(32, 64, kernel_size=3, stride=1, padding=1)  # Reduce to 64 filters

        self.global_pool = nn.AdaptiveAvgPool3d((4, 4, 4))  # Output shape: (64, 4, 4, 4)

        self.fc1 = nn.Linear(64 * 4 * 4 * 4, 128) 
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool(x)
        x = torch.relu(self.conv2(x))
        x = self.pool(x)
        x = torch.relu(self.conv3(x))
        x = self.pool(x)

        x = self.global_pool(x)  # Ensures consistent output shape
        x = torch.flatten(x, start_dim=1)  # Flatten to (batch_size, num_features)
        
        x = torch.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))

        return x


In [67]:
# Check for GPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
# device = "cpu" 

# Initialize model, loss function, and optimizer
model = CNN3D().to(device)
criterion = nn.BCELoss()  # Binary classification loss
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

print("Training complete!")


Epoch [1/5], Loss: 0.7373
Epoch [2/5], Loss: 0.7032
Epoch [3/5], Loss: 0.6363
Epoch [4/5], Loss: 0.6156
Epoch [5/5], Loss: 0.5915
Training complete!


In [68]:
model.eval()
correct, total = 0, 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        predictions = (outputs > 0.5).float()  # Convert probabilities to binary (0 or 1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total * 100
print(f"Test Accuracy: {accuracy:.2f}%")


Test Accuracy: 82.35%
